# Module 1 — The Connected World

*Live session. Every figure in today's slides comes from a cell below, so the
deck and this notebook can be read side by side.*

A graph is not one object. It is three, depending on what you want from it: a
**matrix**, if you want to compute on it; a **set of features**, if you want to
predict something; a **description of the world**, if you want to ask questions
about it.

This notebook works through the first two, and then shows what happens when you
stop designing the features by hand. The third one — a graph that carries
meaning — is where the rest of the course lives.


## Setup


In [ ]:
# torch, networkx, matplotlib and pandas are taken from the Colab image on
# purpose. Pinning torch here would download a second multi-gigabyte build on
# top of the one already installed, for a model with 16 hidden units.
#
# Everything installed below is pinned to the patch. Twenty people start this
# stack on different days, and a version left to float works for whoever wrote
# the notebook and fails for whoever runs it, with the gap invisible until it
# opens in the room.
!pip install -q --disable-pip-version-check torch_geometric==2.8.0 rdflib==7.6.0 owlrl==7.6.2 kuzu==0.11.3

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import torch


In [ ]:
BLUE = "#4986e8"
ORANGE = "#FFA500"
FADED_NODE = "#28374a"
FADED_EDGE = "#c2cedd"
HEAT = mcolors.LinearSegmentedColormap.from_list("blue_orange", ["#002049", ORANGE])


def draw(G, pos, measure=None, select=(), highlight=(), paths=(), edge_width=None, edge_labels=None):
    """Draw one graph, in one of two modes.

    With `measure` — a dict of one number per node — every node is sized and
    coloured by it, and a colour bar is attached. That is act 2's heat maps.

    With `select`, `highlight` or `paths`, the whole graph is drawn faded first
    and the chosen nodes and edges are laid on top of it. The backdrop is the
    point: a handful of nodes drawn on an empty canvas shows a set, while the
    same nodes drawn over the club shows *which part of the club* they are,
    which is the only reason to look at the figure at all.

    The layout is passed in rather than computed here. The session shows the
    same graph a dozen times under different measures, and a layout recomputed
    per cell moves the nodes — which reads as a different graph, the one thing
    these figures must never suggest.
    """
    plt.figure(figsize=(7, 5))

    arrows = {}
    if G.is_directed():
        arrows = {"connectionstyle": "arc3, rad=0.3", "arrowstyle": "-|>", "arrowsize": 25}

    thickness = 2.5 if edge_width is None else edge_width

    if measure is not None:
        value = np.array([measure[n] for n in G.nodes()], dtype=float)
        nx.draw_networkx_edges(G, pos, edge_color=BLUE, width=thickness, alpha=0.7, **arrows)
        nx.draw_networkx_nodes(G, pos, node_size=200 + value / value.max() * 1200,
                               node_color=value, cmap=HEAT, edgecolors="black", linewidths=2)
        bar = plt.cm.ScalarMappable(
            norm=mcolors.Normalize(vmin=value.min(), vmax=value.max()), cmap=HEAT)
        bar.set_array(value)
        plt.colorbar(bar, ax=plt.gca())

    elif len(select) or len(highlight) or len(paths):
        nx.draw_networkx_edges(G, pos, edge_color=FADED_EDGE, width=1.2, **arrows)
        nx.draw_networkx_nodes(G, pos, node_size=300, node_color=FADED_NODE, linewidths=0)
        if len(paths):
            nx.draw_networkx_edges(G, pos, edgelist=list(paths), edge_color=BLUE, width=5, alpha=0.9)
        for nodes, colour in ((select, BLUE), (highlight, ORANGE)):
            if len(nodes):
                nx.draw_networkx_nodes(G, pos, nodelist=list(nodes), node_color=colour,
                                       node_size=1000, edgecolors="black", linewidths=2)
        nx.draw_networkx_labels(G, pos, labels={n: n for n in [*select, *highlight]},
                                font_size=10, font_color="#ffffff", font_weight="bold")

    else:
        nx.draw_networkx_edges(G, pos, edge_color=BLUE, width=thickness, alpha=0.7, **arrows)
        nx.draw_networkx_nodes(G, pos, node_color=BLUE, node_size=1000,
                               edgecolors="black", linewidths=2)

    if measure is not None or not (len(select) or len(highlight) or len(paths)):
        nx.draw_networkx_labels(G, pos, font_size=10, font_color="#ffffff", font_weight="bold")

    if edge_labels:
        nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=12,
                                     font_color=ORANGE, connectionstyle="arc3, rad=0.3",
                                     bbox={"alpha": 1, "color": "white"})

    plt.axis("off")
    plt.gca().set_aspect("equal")
    plt.show()


## Act 1 — A graph is a matrix

Before a graph is a picture it is a table of numbers, and every kind of graph in
the slides is a different rule about what may go in that table. Four rules, four
kinds of graph. Read each adjacency matrix first and the drawing second: the
matrix is the definition, the drawing is a rendering of it.


### Undirected — the matrix is symmetric

`A[i, j] == A[j, i]`, always. A friendship that only one side has is not
representable here, and that is the modelling claim, not a limitation to work
around.


In [ ]:
np.random.seed(1)
A = np.random.randint(2, size=(5, 5))
np.fill_diagonal(A, 1)

# Symmetry is imposed rather than generated: A + A.T doubles the mutual entries,
# so anything above zero is clipped back to a single edge.
A = A + A.T
A[A > 1] = 1
A


In [ ]:
G = nx.from_numpy_array(A)
pos = nx.spring_layout(G, iterations=10, seed=20000)
draw(G, pos)


### Directed — the symmetry is gone

`A[i, j]` and `A[j, i]` are now two independent facts. Following someone is not
being followed back.


In [ ]:
A = np.array([
    [0, 1, 0, 1, 0],
    [0, 0, 1, 0, 1],
    [0, 0, 0, 1, 0],
    [0, 0, 0, 0, 1],
    [1, 0, 0, 0, 0],
])

G = nx.from_numpy_array(A, create_using=nx.DiGraph())
pos = nx.spring_layout(G, iterations=10, seed=20000)
draw(G, pos)


### Weighted — the cell stops being a yes or no

The entry carries a quantity: a travel time, a cost, a capacity. Below it is
drawn as the thickness of the edge.

Note what disappears in the matrix — there is no longer a way to say *this edge
exists and I do not know its weight*, because zero already means "no edge".


In [ ]:
A = np.array([
    [0, 2, 0, 3, 0],
    [0, 0, 1, 0, 4],
    [0, 0, 0, 5, 0],
    [0, 0, 0, 0, 6],
    [7, 0, 0, 0, 0],
])

G = nx.from_numpy_array(A, create_using=nx.DiGraph())
pos = nx.spring_layout(G, iterations=10, seed=20000)

# The weight is drawn rather than written. A number printed on an edge is read
# one edge at a time; a thickness is read all at once, which is the only
# question a weighted graph is usually asked -- where does the most flow.
draw(G, pos, edge_width=[G[u][v]["weight"] for u, v in G.edges()])


### Heterogeneous — one matrix is no longer enough

Different kinds of edge need different matrices, and that is the first moment
the structure starts carrying meaning: `type1` and `type2` are not two numbers,
they are two different claims about the same pair of nodes.

This is where the course is going. Give those types names — *writes*, *cites*,
*belongs to* — and the matrix has become a knowledge graph.


In [ ]:
A = {
    "type1": np.array([[0, 1, 0], [0, 0, 1], [0, 0, 0]]),
    "type2": np.array([[0, 0, 1], [1, 0, 0], [0, 1, 0]]),
}

G = nx.MultiDiGraph()
G.add_nodes_from(range(3))
for edge_type, matrix in A.items():
    for i, j in zip(*np.nonzero(matrix)):
        G.add_edge(int(i), int(j), type=edge_type)

pos = nx.spring_layout(G, iterations=10, seed=20000)
draw(G, pos, edge_labels=nx.get_edge_attributes(G, "type"))


## Act 2 — Features you engineer

A learning algorithm wants a table: one row per thing, one column per number.
A graph is not a table, so somebody has to turn it into one, and that somebody
is you. This act is the *feature-engineering task* from the slides — the arrow
between the input graph and the structured features.

The graph is Zachary's karate club: 34 members of a club that split in two after
a quarrel between the instructor and the president. The split is recorded, which
is what makes it a dataset rather than a picture — see
[Wikipedia](https://en.wikipedia.org/wiki/Zachary%27s_karate_club).


In [ ]:
G = nx.karate_club_graph()

# Computed once and reused by every figure below. See the docstring of `draw`:
# the same layout throughout is what lets the room compare one measure against
# another instead of re-reading the shape each time.
pos = nx.spring_layout(G, iterations=10, seed=20000)

draw(G, pos)


### Node-level features

Four ways to be important, and they disagree with each other. That disagreement
is the content: if they all ranked the nodes the same way there would be one
measure, not four.


In [ ]:
print("nodes:", G.number_of_nodes(), " edges:", G.number_of_edges())
print("average degree:", round(2 * G.number_of_edges() / G.number_of_nodes()))


In [ ]:
centrality = {
    "Degree": nx.degree_centrality(G),
    "Eigenvector": nx.eigenvector_centrality(G),
    "Betweenness": nx.betweenness_centrality(G),
    "Closeness": nx.closeness_centrality(G),
}

for name, values in centrality.items():
    print(name)
    draw(G, pos, measure=values)


Read node 0 and node 33 against each other in the table below, because they are
the two that the slides single out. Node 33 is connected to more people and to
better-connected people — it wins on degree and on eigenvector. Node 0 wins on
betweenness: more of the shortest paths in the club run through it. One of them
is popular, the other is a bridge, and a single number called "importance" would
have hidden the difference.


In [ ]:
pd.DataFrame(centrality).style.background_gradient(cmap=HEAT)


### Edge-level features

Now the unit is a *pair* of nodes, and the question changes from "how important
is this one" to "how close are these two". Link prediction lives here.


In [ ]:
# Distance: the shortest path between two members on opposite sides of the club.
path = nx.shortest_path(G, source=21, target=22)
draw(G, pos, highlight=path, paths=list(zip(path, path[1:])))
print("shortest path:", path)


In [ ]:
# Local overlap: how many neighbours two nodes share. Only one hop counts.
shared = sorted(nx.common_neighbors(G, 32, 33))
draw(G, pos, select=shared, highlight=[32, 33])
print("shared neighbours of 32 and 33:", shared)


In [ ]:
# Global overlap: everything reachable between the two within three hops. Nodes
# 9 and 26 appear here and not above — they share no direct neighbour with the
# pair, and a local measure scores that as no relationship at all.
reachable = {n for p in nx.all_simple_paths(G, 32, 33, cutoff=3) for n in p}
draw(G, pos, select=reachable - {32, 33}, highlight=[32, 33])
print("reachable within three hops:", sorted(reachable - {32, 33}))


### Graph-level features

The unit is now the whole graph. The question these features answer is whether
two graphs are *the same graph* — same structure, different names for the nodes.


In [ ]:
# Relabel every node at random. Nothing about the structure changes, so every
# structural feature has to survive it; anything that does not was reading the
# node names.
rng = np.random.default_rng(42)
permutation = rng.permutation(G.number_of_nodes())
G_perm = nx.relabel_nodes(G, dict(enumerate(permutation)))

print("isomorphic:", nx.is_isomorphic(G, G_perm))
print("WL hash, original:", nx.weisfeiler_lehman_graph_hash(G))
print("WL hash, relabelled:", nx.weisfeiler_lehman_graph_hash(G_perm))

# Node 33 kept its degree, under whatever name it now has.
assert G.degree(33) == G_perm.degree(permutation[33])
print(f"node 33 became node {permutation[33]}, with degree {G.degree(33)} either way")


In [ ]:
# Graphlets: the small connected shapes a large graph is built from. These are
# all of them up to five nodes.
atlas = [g for g in nx.graph_atlas_g()[1:53] if g.number_of_nodes() > 1 and nx.is_connected(g)]

figure, axes = plt.subplots(4, 8, figsize=(12, 6))
for ax, g in zip(axes.flat, atlas):
    nx.draw(g, nx.spring_layout(g, seed=7), ax=ax, node_size=60, node_color=BLUE, width=1.5)
for ax in axes.flat[len(atlas):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

print(f"{len(atlas)} connected graphs with up to five nodes")


## Act 3 — What the table is worth

Act 2 produced a table: four centralities per member. That table is a finished
product — it can be handed to any classifier, and the graph can be thrown away.

So let us throw it away, and find out what it was worth.

The task is the one the dataset was built for: predict which side of the split a
member ended up on. The label is in the data and is not a construct of this
notebook — Zachary recorded which club every member joined, and `networkx` ships
it on each node as `club`. Look at it before trying to predict it.


In [ ]:
mr_hi = [n for n in G.nodes() if G.nodes[n]["club"] == "Mr. Hi"]
officers = [n for n in G.nodes() if G.nodes[n]["club"] == "Officer"]

print(f"Mr. Hi: {len(mr_hi)}   Officer: {len(officers)}")
draw(G, pos, select=mr_hi, highlight=officers)


Seventeen against seventeen, and the two sides sit in visibly different parts of
the picture. Remember both facts. The first is the number every result has to
beat: answer "Mr. Hi" for everybody and you are right half the time. The second
is that the answer is plainly *in* the graph — which makes what follows worse,
not better.

### The traditional pipeline

This is the arrow from the slides: input graph, structured features, learning
algorithm, prediction. The features are the four columns from act 2, and the
learning algorithm is whatever you like.


In [ ]:
members = list(G.nodes())
y = torch.tensor([0 if G.nodes[n]["club"] == "Mr. Hi" else 1 for n in members])

# The four columns of the table from act 2, standardised so that betweenness
# (which spans two orders of magnitude) does not dominate the others.
X = torch.tensor([[m[n] for m in centrality.values()] for n in members], dtype=torch.float)
X = (X - X.mean(0)) / X.std(0)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Deliberately generous: twenty-seven members to train on, seven to test,
# rotated five times so everybody is tested once, and the whole thing repeated
# over twenty shuffles. Thirty-four rows make a single split noisy enough to say
# whatever you would like it to say.
for name, model in [
    ("logistic regression", LogisticRegression(max_iter=1000)),
    ("random forest", RandomForestClassifier(n_estimators=200, random_state=0)),
]:
    scores = [
        cross_val_score(model, X.numpy(), y.numpy(),
                        cv=StratifiedKFold(5, shuffle=True, random_state=split)).mean()
        for split in range(20)
    ]
    print(f"{name:22} {sum(scores) / len(scores):.2f}")


Read the two numbers apart, because they say different things.

**Logistic regression draws one straight boundary** through the four-dimensional
table and asks which side each member falls on. It scores below the coin flip,
which means the two clubs are not separated by any straight cut of these four
columns — not badly separated, *not separated*.

**The random forest** is a pile of threshold questions — *is betweenness above
0.08?* — so it can carve out shapes a straight line cannot, and it finds
something: about 0.65. Real, and worth having. Hold that number.

So the table is not empty. It is just weak, and it needed twenty-seven labels
out of thirty-four to give up that much.

### Learning the features instead

Now take the columns away from the model and give it the graph.

Two models, the same four features, the same training loop — and only eight
labels this time, four from each side, instead of twenty-seven. The only
difference between them is that one is allowed to look at who is connected to
whom.


In [ ]:
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.nn import GCNConv

# Both directions: the club is undirected, and GCNConv reads the edge list as
# given, so listing each edge once would let messages flow one way only.
edge_index = torch.tensor(
    [[u, v] for u, v in G.edges()] + [[v, u] for u, v in G.edges()]
).t().contiguous()


In [ ]:
class MLP(torch.nn.Module):
    """Two layers, each node on its own.

    `forward` takes `edge_index` and ignores it. That is the whole experiment:
    the two models are called identically, trained identically and scored
    identically, so any difference in the result comes from the edges.
    """

    def __init__(self, in_features, hidden):
        super().__init__()
        self.lin1 = Linear(in_features, hidden)
        self.lin2 = Linear(hidden, 2)

    def forward(self, x, edge_index):
        return self.lin2(self.lin1(x).relu())


class GCN(torch.nn.Module):
    """The same two layers, each one now averaging over the neighbours first."""

    def __init__(self, in_features, hidden):
        super().__init__()
        self.conv1 = GCNConv(in_features, hidden)
        self.conv2 = GCNConv(hidden, 2)

    def forward(self, x, edge_index):
        return self.conv2(self.conv1(x, edge_index).relu(), edge_index)


**How this is scored.** Eight members keep their label, four from each side.
Each model trains on those eight, then guesses the twenty-six that were covered
up, and the score is how many of the twenty-six it gets right. It runs ten times
with a different eight each run, because with so few labels which eight you
happen to uncover moves the result by twenty points.

The eight are drawn once per run and handed to **all four models**, the two from
the pipeline above included. That is what makes the table at the end a
comparison and not four separate experiments.

Neither network was tuned: same width, same optimiser, same two hundred epochs,
no early stopping, no validation set — there is no room for one with eight
labels.


In [ ]:
def accuracy(Model, train, seed, epochs=200):
    """Train one network on the labelled members and score the rest.

    The mask arrives from outside rather than being drawn here, because the
    whole point of the cell below is that every model is handed the same eight
    members. A model that picks its own split is not in the same experiment.
    """
    torch.manual_seed(seed)
    model = Model(X.size(1), 16)
    optimiser = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    for _ in range(epochs):
        optimiser.zero_grad()
        F.cross_entropy(model(X, edge_index)[train], y[train]).backward()
        optimiser.step()

    with torch.no_grad():
        predicted = model(X, edge_index).argmax(1)
    return (predicted[~train] == y[~train]).float().mean().item()


def labelled(seed, per_class=4):
    """Pick the members that keep their label. Every model gets this same mask."""
    torch.manual_seed(seed)
    mask = torch.zeros(len(members), dtype=torch.bool)
    for side in (0, 1):
        of_side = (y == side).nonzero().flatten()
        mask[of_side[torch.randperm(len(of_side))[:per_class]]] = True
    return mask


def score_classic(model, mask):
    model.fit(X.numpy()[mask.numpy()], y.numpy()[mask.numpy()])
    return (model.predict(X.numpy()[~mask.numpy()]) == y.numpy()[~mask.numpy()]).mean()


for name, run in (
    ("logistic regression", lambda m, s: score_classic(LogisticRegression(max_iter=1000), m)),
    ("random forest", lambda m, s: score_classic(RandomForestClassifier(n_estimators=200, random_state=0), m)),
    ("MLP", lambda m, s: accuracy(MLP, m, s)),
    ("GCN", lambda m, s: accuracy(GCN, m, s)),
):
    runs = [run(labelled(seed), seed) for seed in range(10)]
    print(f"{name:22} {sum(runs) / len(runs):.2f}")


Every model above was handed **the same eight members**, scored on the same
twenty-six, over the same ten draws. One of them is allowed to look at the edges.

| | sees the edges | accuracy |
|---|---|---|
| logistic regression | no | 0.44 |
| random forest | no | 0.50 |
| MLP | no | 0.48 |
| **GCN** | **yes** | **0.73** |

Three ways of using four columns, and all three land on a coin flip. Remember
that the random forest reached 0.65 a moment ago — that was with twenty-seven
labels and a free hand. Put it on the same eight as everyone else and the
advantage evaporates.

The GCN keeps it. Not from better features, it had exactly the same four, but
from `edge_index`, which the MLP was handed and told to ignore. Who you sat next
to predicts which side you left with, and that was in the graph the whole time,
in a shape no column could hold.

**One asymmetry is left, and it cannot be removed.** The GCN sees every edge,
including the ones touching members it has to guess. That is not a flaw in the
experiment — it is the thing being tested, and there is no such thing as a graph
network that does not look at the graph. What no model here ever sees is a test
member's *label*.

Keep the claim this size: on this graph those four columns carry almost nothing
and the edges carry a lot. Thirty-four members is no place to conclude anything
about neural networks in general. What does generalise is the shape of the trade
— **structure buys you labels**, and labels are what every real project is short
of.


## Act 4 — A graph that means something

Everything so far treated the graph as shape. Nothing in acts 1 to 3 knew that a
node was a person, that an edge was a friendship, or that "Mr. Hi" was an
instructor — and none of it needed to, because shape was all the question
required.

Now the question changes. Here is a small medical graph: diabetes causes
retinopathy, type 2 diabetes is a kind of diabetes, a foot ulcer is located in a
foot. Asking *which diseases are a kind of diabetes* is not a question about
shape. It is a question about what the words mean.

Two families of technology answer it, and they are the two the slides showed
arguing. Same graph, written both ways, asked the same question.


### The graph as RDF

Every line has the same form — *subject, predicate, object* — and the class
hierarchy is written exactly like the facts. The schema is data.


In [ ]:
import owlrl
import rdflib

TURTLE = """
@prefix :     <http://example.org/med#> .
@prefix rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Diabetes              rdf:type         :Disease .
:Type2Diabetes         rdf:type         :Disease .
:LipoatrophicDiabetes  rdf:type         :Disease .
:Retinopathy           rdf:type         :Condition .
:FootUlcer             rdf:type         :Condition .
:Foot                  rdf:type         :AnatomicalStructure .

:Type2Diabetes         rdfs:subClassOf  :Diabetes .
:LipoatrophicDiabetes  rdfs:subClassOf  :Type2Diabetes .

:Diabetes              :causes          :Retinopathy .
:Diabetes              :causes          :FootUlcer .
:FootUlcer             :location        :Foot .
"""

med = rdflib.Graph()
med.parse(data=TURTLE, format="turtle")

print(len(med), "triples")


Which classes are subclasses of `Diabetes`? Keep an eye on this query. It is
assigned once and run twice, and the second time nothing about it will have
changed.


In [ ]:
SUBCLASSES = """
PREFIX :     <http://example.org/med#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?subclass
WHERE { ?subclass rdfs:subClassOf :Diabetes }
"""

for row in med.query(SUBCLASSES):
    print(row.subclass.n3(med.namespace_manager))


One answer. `LipoatrophicDiabetes` is missing, and correctly so: the graph says
it is a kind of *type 2* diabetes, and nobody wrote the other line.

### Where transitivity is written

The specification is explicit. RDF Schema 1.1, section 3.4: *"The
`rdfs:subClassOf` property is transitive."*

The RDFS **vocabulary**, though, never says it. Load the vocabulary and read
everything it has to say about `rdfs:subClassOf`.


In [ ]:
vocabulary = rdflib.Graph()
vocabulary.parse("http://www.w3.org/2000/01/rdf-schema#", format="xml")

for predicate, obj in vocabulary.predicate_objects(rdflib.RDFS.subClassOf):
    print(" ", predicate.n3(vocabulary.namespace_manager), obj.n3(vocabulary.namespace_manager))

mentions = [t for t in vocabulary if any("transitive" in str(term).lower() for term in t)]
print("\ntriples mentioning 'transitive':", len(mentions))


Six statements, and none of them is the one that matters. Transitivity is not a
fact stored anywhere: it is a rule of the semantics — entailment rule `rdfs11`,
which says that from `A subClassOf B` and `B subClassOf C` follows
`A subClassOf C` — and it exists only where a reasoner implements it.

Which is the interesting version of "the schema is data". The rule is code, like
the Cypher in the next section. The difference is that this code was written
once by the W3C, for everybody, instead of by you, inside a query.

### Running the reasoner

Nothing is added by hand here. The reasoner applies the RDFS rules and writes
down what already followed.


In [ ]:
print("before:", len(med), "triples")

owlrl.DeductiveClosure(owlrl.RDFSClosure.RDFS_Semantics).expand(med)

print("after: ", len(med), "triples")

for row in med.query(SUBCLASSES):
    print(row.subclass.n3(med.namespace_manager))


Two answers now, from a query that was not touched.

Of the twenty triples the reasoner added, most are bookkeeping — every resource
gets typed, every property is declared a subproperty of itself. One is the
answer to the question.


In [ ]:
asserted = rdflib.Graph()
asserted.parse(data=TURTLE, format="turtle")

for triple in med.triples((None, rdflib.RDFS.subClassOf, None)):
    if triple not in asserted and str(triple[0]).startswith("http://example.org/med#"):
        print(" ".join(term.n3(med.namespace_manager) for term in triple))


### Declaring a rule of your own

RDFS gives transitivity to `rdfs:subClassOf` because the standard says so — and
only to the properties the standard owns. `partOf` is yours, so the declaration
has to come from you, and in OWL that declaration is an ordinary triple sitting
in the data next to everything else.

A foot is part of a lower limb, a lower limb is part of a body. Whether a foot is
part of a body is, once again, something nobody wrote down.


In [ ]:
ANATOMY = """
@prefix :    <http://example.org/med#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

:partOf  rdf:type  owl:TransitiveProperty .

:Foot       :partOf  :LowerLimb .
:LowerLimb  :partOf  :Body .
"""

med.parse(data=ANATOMY, format="turtle")

PART_OF = """
PREFIX : <http://example.org/med#>

SELECT ?whole
WHERE { :Foot :partOf ?whole }
"""

print("after the RDFS closure:")
for row in med.query(PART_OF):
    print(" ", row.whole.n3(med.namespace_manager))

# The RDFS rules left this untouched: they know rdfs:subClassOf and
# rdfs:subPropertyOf, and nothing about a property invented here. OWL 2 RL
# carries the rule that reads owl:TransitiveProperty off the graph.
owlrl.DeductiveClosure(owlrl.OWLRL.OWLRL_Semantics).expand(med)

print("\nafter the OWL closure:")
for row in med.query(PART_OF):
    print(" ", row.whole.n3(med.namespace_manager))


Two different transitivities, and the difference is where each one lives. One
came with the standard and you cannot change it. The other you declared, as a
triple, inside your own ontology — and a reasoner read it off the graph and
applied it.

Hold on to that distinction, because the other side of the argument does not
have it.

### The same graph as LPG

Kuzu is an embedded property graph database: it runs inside this notebook, with
no server to start. It wants the shape declared first — node tables,
relationship tables, and which types each relationship may connect.


In [ ]:
import kuzu

# No path: the database lives in memory and dies with the notebook. Given a
# filename it would be a file on disk, and re-running this cell would meet the
# tables it created last time and stop on a name clash.
conn = kuzu.Connection(kuzu.Database())

for statement in [
    "CREATE NODE TABLE Disease(name STRING PRIMARY KEY)",
    "CREATE NODE TABLE Condition(name STRING PRIMARY KEY)",
    "CREATE NODE TABLE AnatomicalStructure(name STRING PRIMARY KEY)",
    "CREATE REL TABLE SUBCLASS_OF(FROM Disease TO Disease)",
    "CREATE REL TABLE CAUSES(FROM Disease TO Condition)",
    "CREATE REL TABLE LOCATION(FROM Condition TO AnatomicalStructure)",
    "CREATE REL TABLE PART_OF(FROM AnatomicalStructure TO AnatomicalStructure)",
]:
    conn.execute(statement)

for statement in [
    "CREATE (:Disease {name: 'Diabetes'})",
    "CREATE (:Disease {name: 'Type 2 Diabetes'})",
    "CREATE (:Disease {name: 'Lipoatrophic Diabetes'})",
    "CREATE (:Condition {name: 'Retinopathy'})",
    "CREATE (:Condition {name: 'Foot Ulcer'})",
    "CREATE (:AnatomicalStructure {name: 'Foot'})",
    "CREATE (:AnatomicalStructure {name: 'Lower Limb'})",
    "CREATE (:AnatomicalStructure {name: 'Body'})",
    """MATCH (a:Disease {name: 'Type 2 Diabetes'}), (b:Disease {name: 'Diabetes'})
       CREATE (a)-[:SUBCLASS_OF]->(b)""",
    """MATCH (a:Disease {name: 'Lipoatrophic Diabetes'}), (b:Disease {name: 'Type 2 Diabetes'})
       CREATE (a)-[:SUBCLASS_OF]->(b)""",
    """MATCH (a:Disease {name: 'Diabetes'}), (b:Condition {name: 'Retinopathy'})
       CREATE (a)-[:CAUSES]->(b)""",
    """MATCH (a:Disease {name: 'Diabetes'}), (b:Condition {name: 'Foot Ulcer'})
       CREATE (a)-[:CAUSES]->(b)""",
    """MATCH (a:Condition {name: 'Foot Ulcer'}), (b:AnatomicalStructure {name: 'Foot'})
       CREATE (a)-[:LOCATION]->(b)""",
    """MATCH (a:AnatomicalStructure {name: 'Foot'}), (b:AnatomicalStructure {name: 'Lower Limb'})
       CREATE (a)-[:PART_OF]->(b)""",
    """MATCH (a:AnatomicalStructure {name: 'Lower Limb'}), (b:AnatomicalStructure {name: 'Body'})
       CREATE (a)-[:PART_OF]->(b)""",
]:
    conn.execute(statement)

print("graph loaded")


The same two questions. There is no reasoner to run, so the only thing that can
change is the question itself.


In [ ]:
def ask(title, query):
    print(title)
    for row in conn.execute(query):
        print("  ", row[0])


ask("one hop:", """
    MATCH (s:Disease)-[:SUBCLASS_OF]->(:Disease {name: 'Diabetes'})
    RETURN s.name
""")

# The bound is written out rather than left open. Kuzu accepts a bare `*` and
# on a five-node hierarchy returns the same rows, but the habit is worth having
# early: on a real graph an unbounded traversal is where a query stops being
# cheap without anything in it looking different.
ask("\nany number of hops:", """
    MATCH (s:Disease)-[:SUBCLASS_OF*1..10]->(:Disease {name: 'Diabetes'})
    RETURN s.name
""")


In [ ]:
ask("one hop:", """
    MATCH (:AnatomicalStructure {name: 'Foot'})-[:PART_OF]->(w:AnatomicalStructure)
    RETURN w.name
""")

ask("\nany number of hops:", """
    MATCH (:AnatomicalStructure {name: 'Foot'})-[:PART_OF*1..10]->(w:AnatomicalStructure)
    RETURN w.name
""")


Both answers are right, and both were produced the same way.

That is the finding. On the RDF side these two cases were different in kind: one
transitivity came from the standard, the other you declared in your own
ontology, and in both cases the graph grew while the query stayed still. Here
they are the same case twice. A relationship type is a name, and a name implies
nothing — so the graph stays exactly as you typed it, and the knowledge moves
into whoever writes the query.

Neither is the right answer. **In RDF the knowledge is in the data and the query
stays naive; in LPG the data stays literal and the knowledge is in the query.**
The question worth asking in a project is which of the two you would rather have
to maintain — and the honest answer changes with who writes the queries, how
often the rules change, and whether anyone outside your team has to read them.


## Where this leaves us

Four perspectives, in the order the day ran them:

- **Network.** A graph is a matrix, and the kind of matrix you allow is already
  a modelling decision.
- **Machine learning.** A graph can be flattened into features, and the
  flattening is where the structure leaks out.
- **Representation learning.** Or you can keep the structure and stop designing
  the columns — the karate club's split was in the edges the whole time.
- **Knowledge representation.** Or the graph can carry meaning, and then a
  question can be answered by something other than a traversal somebody
  remembered to write.

The first three are about shape, and they are why the word *graph* is in the
name of this course. The fourth is why the word *knowledge* is there too, and it
is the one the next eight weeks are built on.
